# <span style="font-width:bold; font-size: 3rem; color:#1EB182;">**Garmin Companion**</span><span style="font-width:bold; font-size: 3rem; color:#333;"> - 03: Inference Pipeline</span>

<span style="font-width:bold; font-size: 1.4rem;">One combined deployment, the decision layer, and logging.</span>

> **Not medical advice.** This is a personal training-readiness & recovery monitoring system, not a diagnosis or injury-prediction tool.


We deploy the combined model as a **single KServe deployment** (A8). The predictor does one online `get_feature_vector` round-trip per model, runs all three, and combines them via the §11 max-risk decision layer. Predictions + features are logged via `fv.log` for monitoring.

In [ ]:
!pip install -U 'hopsworks[python]' --quiet

## <span style='color:#ff5f27'>📝 Connect</span>

In [ ]:
import hopsworks

project = hopsworks.login()
mr = project.get_model_registry()
ms = project.get_model_serving()
combined = mr.get_model("garmin_combined", version=1)

## <span style='color:#ff5f27'>🚀 1. Deploy the combined model</span>

`num_instances=1` avoids the KServe scale-to-zero cold start (30–120 s) for a real-time UX. `InferenceLogger(mode='ALL')` captures raw request/response to Kafka; `fv.log` (inside the predictor) captures structured features for monitoring — the two are complementary (A4).

In [ ]:
from hsml.inference_logger import InferenceLogger

# The serving backend resolves the predictor from HopsFS, so upload the local script
# first and pass its absolute /Projects/... path (a bare local path yields
# "Predictor script does not exist").
dataset_api = project.get_dataset_api()
dest_dir = "Resources/deployments/garminrecommendation"
try:
    dataset_api.mkdir(dest_dir)
except Exception:
    pass
uploaded = dataset_api.upload("deployments/predictor.py", dest_dir, overwrite=True)
script_path = uploaded if str(uploaded).startswith("/") else f"/Projects/{project.name}/{dest_dir}/predictor.py"
print("predictor uploaded to:", script_path)

deployment = combined.deploy(
    name="garminrecommendation",
    script_file=script_path,
    resources={"num_instances": 1},
    inference_logger=InferenceLogger(mode="ALL"),
)
deployment.start(await_running=300)

## <span style='color:#ff5f27'>🔮 2. Get a recommendation</span>

In [ ]:
# Daily readiness check. The readiness view is keyed by (user_id, date), so we pass
# both; stress reads the latest realtime state by user_id.
result = deployment.predict(inputs=[{"user_id": "javier", "date": "2024-06-30"}])
print(result)

# After a workout, also pass activity_id to get a recovery-time estimate:
# result = deployment.predict(inputs=[{"user_id": "javier", "date": "2024-06-30",
#                                      "activity_id": "<activity_id>"}])

## <span style='color:#ff5f27'>🪵 3. Inspect the prediction logs</span>

In [ ]:
# Prediction logging via fv.log requires a logging feature group, which this cluster's
# Hive Metastore cannot create (COLUMNS_V2 insert fails server-side). We therefore
# guard the log-inspection step so the pipeline still completes; on a cluster where
# logging FGs work, this block reads back the logged predictions for monitoring.
readiness_fv = combined.get_feature_view()
try:
    readiness_fv.pause_logging()
    readiness_fv.materialize_log(wait=True)
    logged = readiness_fv.read_log()
    print(logged.tail())
except Exception as exc:
    print(f"prediction-log inspection skipped (logging FG unavailable on this cluster): {exc}")

The logged feature group is the bridge to closed-loop monitoring & retraining — we point feature monitoring at it next in **`4_garmin_feature_monitoring.ipynb`**.

For a friendly UI, run `streamlit run streamlit_garmin_app.py`.